# Holt-Winters — otimização de hiperparâmetros

**Etapa:** seleção usando somente treino e validação

A busca compara 15 estruturas de tendência, sazonalidade e amortecimento; os parâmetros de suavização são estimados internamente.

**Entrada:** protocolo, base modelável e origens de validação  
**Resultado principal:** configuração vencedora com MAE de validação 2.9760


## 1. Importações, caminhos e funções

In [ ]:
from pathlib import Path
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from IPython.display import display

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bases/grupo5/gold_daily_modeling.csv").exists())
BASE_PATH = ROOT / "bases/grupo5/gold_daily_modeling.csv"
OUT = ROOT / "analyses/grupo5/outputs"
OUT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = OUT / "protocol.json"
SEED = 42
np.random.seed(SEED)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_data():
    df = pd.read_csv(BASE_PATH, parse_dates=["DATE"]).sort_values("DATE").reset_index(drop=True)
    raw_path = ROOT / "bases/grupo5-tratamento/gold_daily_prices.csv"
    raw = pd.read_csv(raw_path, sep=None, engine="python", parse_dates=["DATE"])
    raw["EXPECTED_TARGET"] = pd.to_numeric(raw["VALUE"], errors="coerce").shift(-1)
    raw["TARGET_DATE"] = raw["DATE"].shift(-1)
    target_lookup = raw[["DATE", "TARGET_DATE", "EXPECTED_TARGET"]]
    df = df.merge(target_lookup, on="DATE", how="left", validate="one_to_one")
    assert df["DATE"].is_monotonic_increasing and not df["DATE"].duplicated().any()
    assert "TARGET_UP" not in df.columns and not df.isna().any().any()
    assert np.allclose(df["TARGET"], df["EXPECTED_TARGET"])
    return df.drop(columns="EXPECTED_TARGET")



In [1]:
def load_protocol(require_data_decisions=False):
    if not PROTOCOL_PATH.exists():
        raise RuntimeError("DECISÃO NECESSÁRIA: execute 01_documentacao_bases.ipynb primeiro.")
    protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if protocol.get("protocol_version") != 2:
        raise RuntimeError("Protocolo desatualizado: execute novamente 01_documentacao_bases.ipynb.")
    if protocol["dataset_sha256"] != sha256(BASE_PATH):
        raise RuntimeError("A base congelada não corresponde ao hash do protocolo.")
    if require_data_decisions:
        missing = [k for k in ("source", "unit", "cleaning", "features", "seasonal_period") if not protocol["decisions"].get(k)]
        if missing:
            raise RuntimeError(f"DECISÃO NECESSÁRIA: complete as decisões {missing} antes de modelar.")
    return protocol

def save_protocol(protocol):
    PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")

def origins(protocol, split):
    return protocol["origins"][split]

def target_dates(df, origin_ids):
    return df["TARGET_DATE"].iloc[origin_ids].to_numpy()


## 2. Funções de ajuste do Holt-Winters

In [ ]:
from itertools import product
from sklearn.ensemble import RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def ml_predict(df, origin, features, model, params):
    train = df.iloc[:origin]; test = df.iloc[[origin]]
    X_train, y_train = train[features], train.TARGET
    X_test = test[features]
    if model == "PLS_Regression":
        scaler = StandardScaler().fit(X_train)
        estimator = PLSRegression(n_components=params["n_components"], scale=False)
        estimator.fit(scaler.transform(X_train), y_train)
        pred = float(estimator.predict(scaler.transform(X_test)).ravel()[0])
    else:
        estimator = RandomForestRegressor(random_state=SEED, n_jobs=-1, **params).fit(X_train, y_train)
        pred = float(estimator.predict(X_test)[0])
    return pred

def ts_predict(df, origin, model, params, exog):
    y = df.GOLD_PRICE.iloc[:origin+1]
    if model == "SARIMAX":
        history_exog = df[exog].iloc[: origin + 1]
        future_exog = df[exog].iloc[[origin]].copy()
        target_date = df["TARGET_DATE"].iloc[origin]
        target_day_of_week = target_date.dayofweek
        target_month = target_date.month
        future_exog.loc[:, "DOW_SIN"] = np.sin(2 * np.pi * target_day_of_week / 7)
        future_exog.loc[:, "DOW_COS"] = np.cos(2 * np.pi * target_day_of_week / 7)
        future_exog.loc[:, "MONTH_SIN"] = np.sin(2 * np.pi * target_month / 12)
        future_exog.loc[:, "MONTH_COS"] = np.cos(2 * np.pi * target_month / 12)
        fitted = SARIMAX(y, exog=history_exog, order=tuple(params["order"]), seasonal_order=tuple(params["seasonal_order"]), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
        return float(fitted.get_forecast(steps=1, exog=future_exog).predicted_mean.iloc[0])
    fitted = ExponentialSmoothing(y, trend=params["trend"], damped_trend=params["damped_trend"], seasonal=params["seasonal"], seasonal_periods=params["seasonal_periods"], initialization_method="estimated").fit(optimized=True)
    smoothing_keys = ["smoothing_level", "smoothing_trend", "smoothing_seasonal", "damping_trend"]
    smoothing = {key: fitted.params.get(key) for key in smoothing_keys}
    smoothing = {key: None if value is None or not np.isscalar(value) else float(value) for key, value in smoothing.items()}
    return float(fitted.forecast(1).iloc[0]), smoothing



In [2]:
def predict_one(df, origin, model, params, protocol):
    if model in {"Random_Forest", "PLS_Regression"}: return ml_predict(df, origin, protocol["decisions"]["features"], model, params)
    return ts_predict(df, origin, model, params, protocol["sarimax_exog"])

def evaluate(df, ids, model, params, protocol):
    started=time.perf_counter(); records=[]
    for origin in ids:
        assert (df["TARGET_DATE"].iloc[:origin] <= df["DATE"].iloc[origin]).all()
        origin_started = time.perf_counter()
        prediction_result = predict_one(df, origin, model, params, protocol)
        if isinstance(prediction_result, tuple):
            prediction, fit_parameters = prediction_result
        else:
            prediction, fit_parameters = prediction_result, {}
        origin_elapsed = time.perf_counter() - origin_started
        actual = float(df.TARGET.iloc[origin])
        records.append({"dataset":"gold", "model":model, "forecast_origin":df.DATE.iloc[origin].date().isoformat(), "target_date":df.TARGET_DATE.iloc[origin].date().isoformat(), "horizon":1, "y_true":actual, "y_pred":prediction, "residual":actual-prediction, "elapsed_seconds":origin_elapsed, "fit_parameters":json.dumps(fit_parameters)})
    output=pd.DataFrame(records); assert np.isfinite(output[["y_true","y_pred","residual"]]).all().all(); assert np.allclose(output.residual, output.y_true-output.y_pred)
    return output, float(mean_absolute_error(output.y_true, output.y_pred)), time.perf_counter()-started


## 3. Busca original de hiperparâmetros

### Espaço de busca

A tabela resume exatamente os candidatos definidos antes do teste final. O critério principal é o MAE walk-forward de validação.

In [ ]:
# As 15 estruturas válidas comparadas na validação
hw_candidates = pd.read_csv(OUT / "tuning_candidates_Holt_Winters.csv")
hw_space = hw_candidates["parameters"].map(json.loads).apply(pd.Series)
hw_space = hw_space[["trend", "seasonal", "damped_trend", "seasonal_periods"]]
hw_space.insert(0, "estrutura", range(1, len(hw_space) + 1))
display(hw_space)

estrutura,trend,seasonal,damped_trend,seasonal_periods
1,add,add,False,5.0
2,mul,mul,False,5.0
3,add,NaN,False,NaN
4,NaN,add,False,5.0
5,add,mul,False,5.0
6,add,mul,True,5.0
7,NaN,mul,False,5.0
8,add,NaN,True,NaN
9,mul,NaN,True,NaN
10,add,add,True,5.0


> **Execução pesada**
>
> Esta célula produziu originalmente os resultados congelados. Ela é mantida para documentar o processo e não deve ser reexecutada apenas para visualizar o notebook.

In [ ]:
MODEL="Holt_Winters"
RUN_TUNING = True
if not RUN_TUNING:
    raise RuntimeError("DECISÃO NECESSÁRIA: confirme a execução pesada e altere RUN_TUNING para True.")
df=load_data(); protocol=load_protocol(require_data_decisions=True)
m = protocol["decisions"]["seasonal_period"]
search = protocol["model_search_spaces"][MODEL]
assert m == 5
assert (df.GOLD_PRICE.iloc[:protocol["splits"]["train_end"]] > 0).all()
candidates = []
for trend, seasonal, damped in product(search["trend"], search["seasonal"], search["damped_trend"]):
    if trend is None and damped:
        continue
    candidates.append({"trend": trend, "seasonal": seasonal, "damped_trend": damped, "seasonal_periods": m if seasonal else None})
assert len(candidates) == 15
progress_path = OUT / f"tuning_progress_{MODEL}.csv"
rows = pd.read_csv(progress_path).to_dict("records") if progress_path.exists() else []
completed = {row["parameters"] for row in rows}
candidate_outputs = {}
for params in candidates:
    key = json.dumps(params, sort_keys=True)
    candidate_path = OUT / f"validation_candidate_{MODEL}_{hashlib.sha256(key.encode()).hexdigest()[:12]}.csv"
    if key in completed and candidate_path.exists():
        candidate_outputs[key] = pd.read_csv(candidate_path)
        continue
    try:
        predictions, mae, elapsed=evaluate(df, origins(protocol,"validation"), MODEL, params, protocol)
        candidate_outputs[key] = predictions
        predictions.to_csv(candidate_path, index=False)
        active_components = int(params["trend"] is not None) + int(params["seasonal"] is not None)
        multiplicative_components = int(params["trend"] == "mul") + int(params["seasonal"] == "mul")
        rows.append({"model":MODEL,"parameters":key,"smoothing":search["smoothing"],"validation_mae":mae,"mae_3":round(mae,3),"active_components":active_components,"damped_complexity":int(params["damped_trend"]),"multiplicative_complexity":multiplicative_components,"mean_seconds_per_origin":predictions.elapsed_seconds.mean(),"elapsed_seconds":elapsed,"status":"ok"})
    except Exception as exc:
        rows.append({"model":MODEL,"parameters":key,"validation_mae":np.nan,"elapsed_seconds":np.nan,"status":f"failed: {type(exc).__name__}"})
    pd.DataFrame(rows).to_csv(progress_path, index=False)
ranking=pd.DataFrame(rows).sort_values(["mae_3","active_components","damped_complexity","multiplicative_complexity","mean_seconds_per_origin"], na_position="last"); display(ranking)
ranking.to_csv(OUT / f"tuning_candidates_{MODEL}.csv", index=False)
winner = json.loads(ranking.iloc[0]["parameters"])
candidate_outputs[json.dumps(winner, sort_keys=True)].to_csv(OUT / f"validation_predictions_{MODEL}.csv", index=False)
print("Tuning concluído. Revise o ranking antes de executar o bloco de aceite.")

## 4. Parâmetros vencedores congelados

In [ ]:
MODEL = "Holt_Winters"
ACCEPTED_PARAMS = {"damped_trend": False, "seasonal": "add", "seasonal_periods": 5, "trend": "add"}
protocol = load_protocol(require_data_decisions=True)
ranking = pd.read_csv(OUT / f"tuning_candidates_{MODEL}.csv")
winner = json.loads(ranking.iloc[0]["parameters"])
display(ranking.head())
if ACCEPTED_PARAMS is None:
    raise RuntimeError("DECISÃO NECESSÁRIA: revise o ranking e preencha ACCEPTED_PARAMS; o teste final continua bloqueado.")
if json.dumps(ACCEPTED_PARAMS, sort_keys=True) != json.dumps(winner, sort_keys=True):
    raise ValueError("ACCEPTED_PARAMS deve ser o vencedor do critério fixado antes do tuning.")
protocol["decisions"]["accepted_parameters"][MODEL] = ACCEPTED_PARAMS
save_protocol(protocol)
selected = ranking.iloc[0]
selected_record = pd.DataFrame([{"dataset": "gold", "model": MODEL, "parameters": json.dumps(ACCEPTED_PARAMS, sort_keys=True), "validation_mae": selected.validation_mae, "mean_seconds_per_origin": selected.mean_seconds_per_origin, "n_candidates": len(ranking), "selection_criterion": json.dumps(protocol["selection_criterion"], ensure_ascii=False)}])
hyperparameters_path = OUT / "hyperparameters.csv"
if hyperparameters_path.exists():
    previous = pd.read_csv(hyperparameters_path)
    selected_record = pd.concat([previous[previous.model != MODEL], selected_record], ignore_index=True)
selected_record.to_csv(hyperparameters_path, index=False)
print("Parâmetros aceitos e teste final liberado apenas para este modelo.")

## Resultado congelado

In [ ]:
# Leitura leve do ranking congelado — não executa tuning
ranking_congelado = pd.read_csv(OUT / "tuning_candidates_Holt_Winters.csv")
ranking_congelado = ranking_congelado.sort_values("validation_mae").head(5)
display(ranking_congelado[['parameters', 'validation_mae', 'mean_seconds_per_origin', 'status']])

parameters,validation_mae,mean_seconds_per_origin,status
"{""damped_trend"": false, ""seasonal"": ""add"", ""seasonal_periods"": 5, ""trend"": ""add""}",2.976001,1.125441,ok
"{""damped_trend"": false, ""seasonal"": ""mul"", ""seasonal_periods"": 5, ""trend"": ""mul""}",2.976083,3.490514,ok
"{""damped_trend"": false, ""seasonal"": null, ""seasonal_periods"": null, ""trend"": ""add""}",2.977133,0.216642,ok
"{""damped_trend"": true, ""seasonal"": ""mul"", ""seasonal_periods"": 5, ""trend"": ""add""}",2.978529,2.168297,ok
"{""damped_trend"": false, ""seasonal"": ""add"", ""seasonal_periods"": 5, ""trend"": null}",2.978801,0.112562,ok


O primeiro candidato é o vencedor congelado do Holt-Winters. A tabela mostra somente os cinco melhores para facilitar a leitura.

## Interpretação

O vencedor foi escolhido pelo menor MAE walk-forward de validação. O teste final não participou da seleção. O resultado está congelado e a busca não precisa ser reexecutada.

## Artefatos

Arquivos usados ou produzidos nesta etapa:

- `outputs/tuning_candidates_Holt_Winters.csv`
- `outputs/hyperparameters.csv`
- `outputs/validation_predictions_Holt_Winters.csv`